# Legal BPE tokenizer (50K) - CS-781 Phase 1

Byte-level BPE trained with the HF `tokenizers` (Rust) library on a stratified sample:
**10% Supreme Court / 50% High Courts / 35% Tribunals / 5% Acts**.

Cells: install -> config -> sampling helpers -> sample corpus (cached) -> train -> evaluate -> wrap for `transformers`.

In [ ]:
%pip install -q tokenizers ijson huggingface_hub transformers

In [ ]:
import os, io, json, time, random, zlib, unicodedata
from collections import defaultdict
from huggingface_hub import hf_hub_download, login

# ---------------- paths ----------------
OUTPUT_DIR = "/kaggle/working/cs781_phase1_v2" if os.path.exists("/kaggle/working") else "./cs781_phase1_v2"
DATA_DIR   = os.path.join(OUTPUT_DIR, "data")
TOK_DIR    = os.path.join(OUTPUT_DIR, "tokenizer")
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(TOK_DIR, exist_ok=True)
CORPUS_PATH = os.path.join(DATA_DIR, "bpe_train_sample.jsonl")   # cached sampled corpus
EVAL_PATH   = os.path.join(DATA_DIR, "bpe_heldout.jsonl")        # held-out docs (never seen by BPE)

# ---------------- data ----------------
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCES = {
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "high_courts":   "Phase_1_Pretraining/Train/train_high_courts.json",
    "tribunals":     "Phase_1_Pretraining/Train/train_tribunals.json",
    "acts":          "Phase_1_Pretraining/Train/train_acts.json",
}
MIX = {"supreme_court": 0.10, "high_courts": 0.50, "tribunals": 0.35, "acts": 0.05}   # share of sampled CHARACTERS
assert abs(sum(MIX.values()) - 1.0) < 1e-9

TOTAL_SAMPLE_CHARS = 500_000_000   # ~0.5 GB of text. Raise to 1e9 if RAM allows (BPE training is RAM-bound, not time-bound)
OVERSAMPLE         = 1.3           # >1 => we hit the quota before the end of each file, sampling still spread over ~77%+ of it
MIN_DOC_CHARS      = 100
SEED               = 42
EVAL_BUCKET_PCT    = 1             # ~1% of docs (by content hash) are reserved for evaluation only
EVAL_CHARS_PER_SOURCE = 3_000_000

# ---------------- tokenizer ----------------
VOCAB_SIZE     = 50_000
SPLIT_DIGITS   = True    # 302 -> "3","0","2" (better number generalisation, ~few % longer sequences). False = GPT-2 style digit runs
LOWERCASE      = False   # keep case: "Act", "Section", "IPC", names carry signal for a legal LM
SPECIAL_TOKENS = ["<|pad|>", "<|unk|>", "<|endoftext|>", "<|bos|>", "<|mask|>"]   # first 3 ids (0,1,2) identical to your old tokenizer

# ---------------- HF auth (never hard-code the token) ----------------
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF")
    except Exception:
        HF_TOKEN = None
if HF_TOKEN:
    login(token=HF_TOKEN)


In [ ]:
import ijson
print("ijson backend:", ijson.backend, "(want 'yajl2_c'; pure-python 'python' backend is ~50x slower)")

def local_path(name):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=SOURCES[name], repo_type="dataset", token=HF_TOKEN)

def iter_json_docs(path):
    # Streams records from a huge file WITHOUT loading it into RAM.
    # Handles: top-level JSON array, top-level {key: record} dict, or JSONL.
    with open(path, "rb") as f:
        first = b""
        while True:
            ch = f.read(1)
            if not ch:
                return
            if not ch.isspace():
                first = ch
                break
        f.seek(0)
        if first == b"[":
            yield from ijson.items(f, "item")
            return
        line = f.readline(2_000_000)
        is_jsonl = False
        if line.endswith(b"\n"):
            try:
                json.loads(line); is_jsonl = True
            except Exception:
                pass
        f.seek(0)
        if is_jsonl:
            for ln in f:
                ln = ln.strip()
                if ln:
                    yield json.loads(ln)
        else:
            for _, v in ijson.kvitems(f, ""):
                yield v

TEXT_KEYS = ("text", "content", "body", "judgment", "full_text")
def extract_text(rec):
    if isinstance(rec, str):
        return rec
    if isinstance(rec, dict):
        for k in TEXT_KEYS:
            v = rec.get(k)
            if isinstance(v, str):
                return v
        strs = [v for v in rec.values() if isinstance(v, str)]
        return max(strs, key=len) if strs else ""
    return ""

def doc_bucket(text):
    # deterministic 0..99 bucket from content -> same doc always lands in the same split, even across sources
    return zlib.crc32(text[:2000].encode("utf-8", "ignore")) % 100

def sample_source(name, quota_chars, rng, train_fh, eval_fh):
    path = local_path(name)
    approx_chars = os.path.getsize(path) * 0.95            # chars ~ bytes (rough; JSON overhead + multibyte roughly cancel)
    p = min(1.0, OVERSAMPLE * quota_chars / approx_chars)
    got = n_docs = ev_chars = n_seen = 0
    t0 = time.time()
    for rec in iter_json_docs(path):
        n_seen += 1
        text = extract_text(rec)
        if len(text) < MIN_DOC_CHARS:
            continue
        if doc_bucket(text) < EVAL_BUCKET_PCT:              # held-out docs never go to BPE training
            if ev_chars < EVAL_CHARS_PER_SOURCE:
                eval_fh.write(json.dumps({"src": name, "text": text}, ensure_ascii=False) + "\n")
                ev_chars += len(text)
            continue
        if rng.random() > p:
            continue
        train_fh.write(json.dumps({"src": name, "text": text}, ensure_ascii=False) + "\n")
        got += len(text); n_docs += 1
        if n_docs % 5000 == 0:
            print(f"  {name}: {got/1e6:.0f}/{quota_chars/1e6:.0f} MB  ({n_seen} records scanned, {time.time()-t0:.0f}s)")
        if got >= quota_chars:
            break
    return got, n_docs, ev_chars


In [ ]:
if os.path.exists(CORPUS_PATH) and os.path.exists(EVAL_PATH):
    print("Sample corpus already built -> reusing", CORPUS_PATH)
else:
    rng = random.Random(SEED)
    tmp_train, tmp_eval = CORPUS_PATH + ".part", EVAL_PATH + ".part"     # write to .part, rename at the end: no half-built caches
    summary = {}
    with open(tmp_train, "w", encoding="utf-8") as tf, open(tmp_eval, "w", encoding="utf-8") as ef:
        for name, frac in MIX.items():
            quota = int(TOTAL_SAMPLE_CHARS * frac)
            print(f"\n[{name}] target {quota/1e6:.0f} MB ({frac:.0%})")
            got, n, ev = sample_source(name, quota, rng, tf, ef)
            summary[name] = (got, n, ev)
            if got < 0.9 * quota:
                print(f"  WARNING: only got {got/1e6:.0f} MB of {quota/1e6:.0f} MB for {name} - mix will be off. "
                      f"Check the text key / raise OVERSAMPLE.")
    os.replace(tmp_train, CORPUS_PATH); os.replace(tmp_eval, EVAL_PATH)

    tot = sum(v[0] for v in summary.values())
    print("\nRealised mix (by characters):")
    for k, (got, n, ev) in summary.items():
        print(f"  {k:14s} {got/1e6:8.1f} MB  {got/tot:6.1%}  (target {MIX[k]:.0%})  docs={n}")


In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers, processors, Regex

# GPT-2 style split pattern; \p{N} (single digit) vs \p{N}+ (digit run) controlled by SPLIT_DIGITS.
# The optional leading space stays attached to the first digit, so "Section 302" -> "ĠSection","Ġ3","0","2" (no lone-space token).
num = r"\p{N}" if SPLIT_DIGITS else r"\p{N}+"
SPLIT_PATTERN = r"'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?" + num + r"| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"

tok = Tokenizer(models.BPE(unk_token=None))          # byte-level => every string is representable, <|unk|> is never produced
norms = [normalizers.NFC()] + ([normalizers.Lowercase()] if LOWERCASE else [])
tok.normalizer = normalizers.Sequence(norms)
tok.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.Split(Regex(SPLIT_PATTERN), behavior="isolated"),
    pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False),
])
tok.decoder = decoders.ByteLevel()
tok.post_processor = processors.ByteLevel(trim_offsets=False)

trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=SPECIAL_TOKENS,                    # get ids 0..4 in this order
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),   # all 256 bytes always in vocab
    show_progress=True,
)

def corpus_iter(batch_size=1000):
    buf = []
    with open(CORPUS_PATH, encoding="utf-8") as f:
        for line in f:
            buf.append(json.loads(line)["text"])
            if len(buf) == batch_size:
                yield buf; buf = []
    if buf:
        yield buf

t0 = time.time()
tok.train_from_iterator(corpus_iter(), trainer=trainer)
print(f"Training done in {(time.time()-t0)/60:.1f} min | vocab size = {tok.get_vocab_size()}")
if tok.get_vocab_size() < VOCAB_SIZE:
    print("WARNING: vocab smaller than target - sample corpus too small / min_frequency too high.")

tok.save(os.path.join(TOK_DIR, "tokenizer.json"))
tok.model.save(TOK_DIR)                                # also writes vocab.json + merges.txt
meta = {
    "vocab_size": tok.get_vocab_size(),
    "pad_id": tok.token_to_id("<|pad|>"), "unk_id": tok.token_to_id("<|unk|>"), "eos_id": tok.token_to_id("<|endoftext|>"),
    "bos_id": tok.token_to_id("<|bos|>"), "mask_id": tok.token_to_id("<|mask|>"),
    "split_digits": SPLIT_DIGITS, "lowercase": LOWERCASE, "mix": MIX, "sample_chars": TOTAL_SAMPLE_CHARS,
}
with open(os.path.join(DATA_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)
print(meta)


In [ ]:
# ---------- 1) held-out efficiency per source ----------
held = defaultdict(list)
with open(EVAL_PATH, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line); held[r["src"]].append(r["text"])

unk_id = tok.token_to_id("<|unk|>")
print(f"{'source':14s} {'docs':>6s} {'chars/token':>12s} {'tokens/word':>12s} {'unk':>5s}")
for src, docs in held.items():
    encs = tok.encode_batch(docs)
    n_tok = sum(len(e.ids) for e in encs)
    n_chr = sum(len(d) for d in docs)
    n_wrd = sum(len(d.split()) for d in docs)
    n_unk = sum(e.ids.count(unk_id) for e in encs)
    print(f"{src:14s} {len(docs):6d} {n_chr/n_tok:12.2f} {n_tok/n_wrd:12.2f} {n_unk:5d}")

# ---------- 2) lossless round-trip (vs NFC-normalised original) ----------
bad = 0; checked = 0
for src, docs in held.items():
    for d in docs[:100]:
        checked += 1
        if tok.decode(tok.encode(d).ids, skip_special_tokens=False) != unicodedata.normalize("NFC", d):
            bad += 1
print(f"\nRound-trip mismatches: {bad}/{checked}  (should be 0)")

# ---------- 3) eyeball legal phrases ('Ġ' = leading space) ----------
probes = [
    "The Hon'ble Supreme Court held that Section 302 of the Indian Penal Code, 1860 applies.",
    "Writ Petition (Civil) No. 1234 of 2019 under Article 226 of the Constitution of India.",
    "anticipatory bail under Section 438 Cr.P.C. was rejected by the Sessions Judge.",
    "The assessee preferred an appeal before the Income Tax Appellate Tribunal, Mumbai Bench.",
]
for p in probes:
    t = tok.encode(p).tokens
    print(f"\n{len(t)} tokens: {t}")

# ---------- 4) vocab health ----------
vocab = tok.get_vocab()
by_id = sorted(vocab.items(), key=lambda kv: kv[1])
print("\nLast 25 learned merges (rarest):", [t for t, _ in by_id[-25:]])


In [ ]:
from transformers import PreTrainedTokenizerFast

hf_tok = PreTrainedTokenizerFast(
    tokenizer_file=os.path.join(TOK_DIR, "tokenizer.json"),
    pad_token="<|pad|>", unk_token="<|unk|>", eos_token="<|endoftext|>",
    bos_token="<|bos|>", mask_token="<|mask|>",
)
hf_tok.save_pretrained(TOK_DIR)
print(len(hf_tok), hf_tok("Section 302 IPC")["input_ids"])
print("Reload with: PreTrainedTokenizerFast.from_pretrained(", repr(TOK_DIR), ")")

# Tip: for GPU efficiency set the model's embedding size to a multiple of 64, e.g. 50_048 (tokenizer stays 50_000).
